# Sanity — is the cluster actually serving, and through the gateway?

Run this **before any load test**. It answers six questions in order, and
each one makes the next meaningful:

1. what answers at all
2. what model the fleet is serving
3. how much KV each pod got, and what concurrency that buys
4. does a request really traverse the gateway
5. is the KV hop moving bytes or recording a row
6. was anything scarce

It writes `../reports/sanity-<date>.pdf`.

**Through the tunnel** (`make ssh`) from a laptop, or straight on the box.
Latency here is indicative only — the tunnel adds your round-trip. Use
`make load` for numbers you intend to quote.


In [ ]:
from datetime import datetime, timezone
from utils import (Cluster, Report, sum_by_name, filler, percentile,
                   grafana_dashboard, grafana_explore, REPORTS)

cluster = Cluster()   # 127.0.0.1:8080 gateway, :30800 prefill, :30801 decode
report = Report('Dodona — cluster sanity', 'one pass, no load')
print('reports go to', REPORTS)


## 1. What answers

Everything below is meaningless if the gateway is down, and a `False` on
`decode /metrics` means the pool never came up rather than that it is idle.


In [ ]:
report.heading('1. Endpoints')
up = cluster.endpoints()
for name, ok in up.items():
    report.verdict(ok, name)

if not up.get('gateway /metrics'):
    report.text('The gateway is unreachable. Is the tunnel open (make ssh)?')
    report.text('Is orch-serve running?   sudo k3s kubectl get pods')


## 2. What is it serving

The model id the gateway advertises has to match what the pods were started
with, or vLLM rejects every request with an unknown model.


In [ ]:
report.heading('2. Model and pools')
model = cluster.served_model()
report.verdict(bool(model), f'gateway advertises: {model}')

rows = []
for pool in ('prefill', 'decode'):
    m = cluster.pool_metrics(pool)
    rows.append([pool,
                 int(sum_by_name(m, 'vllm:num_requests_running')),
                 int(sum_by_name(m, 'vllm:num_requests_waiting')),
                 f"{int(sum_by_name(m, 'vllm:prompt_tokens_total')):,}",
                 f"{int(sum_by_name(m, 'vllm:generation_tokens_total')):,}"])
report.table(['pool', 'running', 'waiting', 'prompt tok', 'gen tok'], rows,
             note='prompt tok 0 on a pool means nothing was ever routed to it.')

# Prefill should show almost no generation: its leg is sent with max_tokens=1.
gen_prefill = rows[0][4].replace(',', '')
report.verdict(int(gen_prefill) < 100,
               f'prefill generated {rows[0][4]} tokens (expected near 0, max_tokens=1)')


## 3. Capacity — what each pod actually got

`kv_cache_usage_perc` is a **ratio of that pod's own KV**, so it says nothing
about absolute size. The absolute number is in the pod's startup log:

```bash
make logs | grep -E 'Available KV cache memory|Maximum concurrency'
```

Fill the two values in below from that line. They are the denominator of
every concurrency claim in the submission, and guessing them is how a
whole-card figure ends up quoted for a slice.


In [ ]:
# FROM THE POD LOG — replace with what YOUR pods report:
#   make logs | grep -E 'Available KV cache memory|Maximum concurrency'
KV_GIB_PER_POD  = 4.3     # 'Available KV cache memory: X GiB'
SEQ_PER_POD     = 4.9     # 'Maximum concurrency for 16,384 tokens: Xx'
DECODE_REPLICAS = 1       # KEDA scales this to 2 under load

report.heading('3. Capacity')
pool_total = SEQ_PER_POD * DECODE_REPLICAS
report.table(['quantity', 'value', 'where it comes from'], [
    ['KV per decode pod', f'{KV_GIB_PER_POD:.2f} GiB', 'pod log: Available KV cache memory'],
    ['full-length seqs / pod', f'{SEQ_PER_POD:.1f}', 'pod log: Maximum concurrency'],
    ['decode replicas', DECODE_REPLICAS, 'kubectl get deploy vllm-decode'],
    ['decode pool concurrency', f'{pool_total:.1f}', 'seqs/pod x replicas'],
], note='A pool this small is the point: a policy needs something to be scarce.')
report.verdict(pool_total < 20,
               f'pool of {pool_total:.1f} is small enough for KV to bind')


## 4. Does a request traverse the gateway

This is the one that decides whether any of the control plane is real.
`orch_requests_total` only moves if the request went through our gateway —
a completion that reached a pod directly leaves it at zero, and from outside
the two look identical.


In [ ]:
report.heading('4. Does a request traverse the gateway')
before = sum_by_name(cluster.gateway_metrics(), 'orch_requests_total')
answer = cluster.chat('Reply with the single word: ok', max_tokens=8)
after  = sum_by_name(cluster.gateway_metrics(), 'orch_requests_total')

report.verdict(after > before,
               f'orch_requests_total moved {before:.0f} -> {after:.0f}')
report.verdict('_error' not in answer,
               f"completion returned in {answer['_latency_s']}s")

if '_error' in answer:
    report.text(f"ERROR {answer['_error']}: {answer.get('_body','')[:200]}")
else:
    said = ((answer.get('choices') or [{}])[0].get('message') or {}).get('content')
    report.table(['field', 'value'], [
        ['content', repr(said)],
        ['via', answer.get('via')],
        ['latency', f"{answer['_latency_s']}s  (tunnel included)"],
    ], note='via=local means our cluster served it; via=overflow means it was bursted out.')


## 5. Is the KV hop real

`kv_hop` in the response body proves the router **recorded** a hop. It does
not prove bytes moved — that needs the connector, and the connector reports
its transport in the pod log:

```bash
make logs | grep -E 'HCAs|transport|Mooncake'
```

| what the log says | what it means |
|---|---|
| `Found 0 HCAs` · `Using TCP transport` | works, over loopback TCP, not RDMA |
| `Found 1 HCAs` · RDMA transport | the fast path |
| no Mooncake lines at all | no connector: decode re-prefills, the hop is a row |


In [ ]:
# FROM THE POD LOG — make logs | grep -E 'HCAs|transport|Mooncake'
#   'rdma'  Found >=1 HCAs, RDMA transport      <- the fast path
#   'tcp'   Found 0 HCAs, Using TCP transport   <- works, slower
#   'none'  no Mooncake lines at all            <- decode re-prefills
MOONCAKE_TRANSPORT = 'unknown'

report.heading('5. The KV hop')
hop = answer.get('kv_hop') if isinstance(answer, dict) else None
report.verdict(bool(hop), 'the router recorded a hop')
if hop:
    report.table(['field', 'value'], [[k, v] for k, v in hop.items()],
                 note='This proves a hop was RECORDED. It does not prove bytes moved.')

report.verdict(MOONCAKE_TRANSPORT in ('rdma', 'tcp'),
               f'connector transport is known: {MOONCAKE_TRANSPORT}')
if MOONCAKE_TRANSPORT == 'tcp':
    report.text('TCP: /dev/infiniband is not visible in the pod. Blocks move,')
    report.text('but not at RDMA speed.')
elif MOONCAKE_TRANSPORT == 'none':
    report.text('No connector: decode re-prefills and the hop is only a record.')


### Does decode re-prefill?

A cheap, external signal. If the connector moves blocks, decode should NOT
have to compute the prompt — so `prefix_cache_hits_total / queries_total` on
the decode pool rises, and its time-to-first-token on a long prompt collapses.

`prompt_tokens_total` being equal across pools is suggestive but not proof:
vLLM counts the tokens in the request whether or not they were cached. The
prefix-cache ratio is the honest one.


In [ ]:
report.heading('5b. Does decode re-prefill?')
rows = []
for pool in ('prefill', 'decode'):
    m = cluster.pool_metrics(pool)
    hits = sum_by_name(m, 'vllm:prefix_cache_hits_total')
    qs   = sum_by_name(m, 'vllm:prefix_cache_queries_total')
    rows.append([pool, f"{int(sum_by_name(m, 'vllm:prompt_tokens_total')):,}",
                 f'{int(hits):,}', f'{int(qs):,}',
                 f'{hits / qs:.1%}' if qs else 'n/a'])
report.table(['pool', 'prompt tok', 'prefix hits', 'queries', 'hit rate'], rows,
             note='Decode loading blocks rather than recomputing shows as a high hit rate.')

pre = int(rows[0][1].replace(',', ''))
dec = int(rows[1][1].replace(',', ''))
same = pre and dec and abs(pre - dec) < max(pre, dec) * 0.05
report.verdict(not same,
               'decode processes fewer prompt tokens than prefill')
if same:
    report.text(f'Both pools report ~{pre:,} prompt tokens. queue.py sends the same')
    report.text('prompt to both legs, so this is expected — and it means decode is')
    report.text('NOT skipping prompt work. Confirm with TTFT on a long prompt.')


## 6. Was anything scarce

The question that separates a result from a number. If KV never rises, an
admission policy reading it is **inert** — installed, running on every
request, unable to change any outcome.


In [ ]:
report.heading('6. Was anything scarce')
kv = cluster.kv_usage()
report.table(['pool', 'kv_cache_usage_perc', 'vs 0.90 threshold'],
             [[p, f'{v:.4f}', 'BELOW' if v < 0.90 else 'CROSSED'] for p, v in kv.items()],
             note='Idle cluster: near zero is expected. make load is what moves this.')
report.text('An idle reading is not a verdict on the policy — see make load.')
report.bar('KV usage vs the 0.90 admission threshold', list(kv) or ['none'],
           {'kv_cache_usage_perc': [kv.get(p, 0) for p in (kv or ['none'])]},
           ylabel='ratio', hline=0.90,
           note='Dashed line is the kv_threshold trigger. Below it the policy is inert.')


## 7. Charts — where the work actually lands

Three claims the architecture makes, each with the chart that would falsify it.

| claim | falsified if |
|---|---|
| prefill builds KV, decode generates | prefill's generation count is not ~0 |
| the slice, not the card, sets concurrency | the KV bar matches 64 GiB, not ~4.3 |
| decode loads blocks instead of recomputing | both pools show the same prompt tokens |


In [ ]:
pre_m = cluster.pool_metrics('prefill')
dec_m = cluster.pool_metrics('decode')

report.heading('7. Where the work lands')
report.bar('Who does which work', ['prefill', 'decode'], {
    'prompt tokens':     [sum_by_name(m, 'vllm:prompt_tokens_total') for m in (pre_m, dec_m)],
    'generation tokens': [sum_by_name(m, 'vllm:generation_tokens_total') for m in (pre_m, dec_m)],
}, ylabel='tokens (log)', log=True,
   note='Prefill generating ~0 is the max_tokens=1 leg working. Equal PROMPT bars '
        'mean decode recomputed the prompt — the hop moved no blocks.')

report.bar('Prefix cache hit rate', ['prefill', 'decode'], {
    'hit rate': [sum_by_name(m, 'vllm:prefix_cache_hits_total') /
                 max(sum_by_name(m, 'vllm:prefix_cache_queries_total'), 1)
                 for m in (pre_m, dec_m)],
}, ylabel='hits / queries',
   note='A decode pool loading blocks from the connector should sit well above prefill.')


### The capacity budget

Why the pool is ~4.9 sequences and not 73.5. The whole card has 64 GiB of KV;
a 12 GiB slice has ~4.3 after weights and CUDA graphs. Quoting the card's
figure for a slice overstates concurrency by 15x.


In [ ]:
SLICE_GIB   = 12.0
UTIL        = 0.90
WEIGHTS_GIB = 5.2      # Qwen2.5-7B AWQ int4, embeddings left in fp16
overhead_gib = SLICE_GIB * UTIL - WEIGHTS_GIB - KV_GIB_PER_POD

report.heading('7b. The slice budget')
report.table(['component', 'GiB'], [
    ['slice x util', f'{SLICE_GIB * UTIL:.2f}'],
    ['weights', f'{WEIGHTS_GIB:.2f}'],
    ['activations + CUDA graphs', f'{overhead_gib:.2f}'],
    ['KV cache', f'{KV_GIB_PER_POD:.2f}'],
], note='Overhead is budget minus weights minus KV. Measured, not predicted.')
report.bar('Where the 12 GiB slice goes', ['weights', 'activations + graphs', 'KV cache'],
    {'GiB': [WEIGHTS_GIB, max(overhead_gib, 0), KV_GIB_PER_POD]}, ylabel='GiB',
    note=f'KV is {KV_GIB_PER_POD:.2f} of {SLICE_GIB * UTIL:.2f} usable GiB — which is why the '
         f'pool is ~{SEQ_PER_POD:.1f} and not the 73.5 the whole card would suggest.')


## 8. Latency — a distribution, not one number

Ten sequential short requests through the gateway, first discarded as warmup.

**This includes the ssh tunnel**, so treat it as a shape rather than a
figure to quote. `make load` runs on the box for the real thing.


In [ ]:
report.heading('8. Latency, short prompt')
lat = cluster.sample('Reply with the single word: ok', n=10, max_tokens=8)
if lat:
    p50, p95 = percentile(lat, .5), percentile(lat, .95)
    report.table(['metric', 'seconds'], [
        ['n', len(lat)], ['min', f'{min(lat):.3f}'], ['p50', f'{p50:.3f}'],
        ['p95', f'{p95:.3f}'], ['max', f'{max(lat):.3f}'],
    ], note='Through the tunnel. Shape only — use make load to quote numbers.')
    report.line('Latency, 10 sequential short requests',
                list(range(1, len(lat) + 1)), {'seconds': lat},
                xlabel='request', ylabel='seconds',
                hlines={f'p50 {p50:.3f}s': p50, f'p95 {p95:.3f}s': p95},
                note='A rising line means something is accumulating; flat is healthy.')
else:
    report.verdict(False, 'no successful samples — see section 4')


## 9. The decisive hop test — latency vs prompt length

**This is the one that settles whether blocks move.**

Decode is sent the full prompt either way. The difference:

- **blocks moved** → decode loads KV, so its cost barely grows with prompt length
- **no blocks** → decode re-prefills, so latency grows roughly linearly with it

A clear upward slope here means the hop is still bookkeeping, whatever the
`kv_hop` record says.


In [ ]:
report.heading('9. Latency vs prompt length')
LENGTHS = [256, 1024, 4096, 8192, 12288]
points, dropped = [], []
for n_tok in LENGTHS:
    prompt = filler(n_tok) + '\n\nReply with the single word: ok'
    cluster.last_error = None
    s = cluster.sample(prompt, n=3, max_tokens=8, warmup=1)
    if s:
        points.append((n_tok, percentile(s, .5)))
    else:
        dropped.append((n_tok, cluster.last_error or 'no samples'))

report.table(['prompt tokens', 'p50 seconds'],
             [[f'{n:,}', f'{v:.3f}'] for n, v in points],
             note='Warmup discarded, so these are cache-warm numbers.')

# A sweep that silently skips the lengths that failed draws a flat line
# and calls it a result. 12,000 tokens IS the use case, so losing it
# is not a detail.
report.verdict(not dropped, f'all {len(LENGTHS)} prompt lengths completed')
for n_tok, why in dropped:
    report.text(f'{n_tok:,} tokens DROPPED — {why}')

if len(points) >= 2:
    growth = points[-1][1] / max(points[0][1], 1e-6)
    span = points[-1][0] / max(points[0][0], 1)
    report.verdict(growth < span / 4,
                   f'latency grew {growth:.1f}x while the prompt grew {span:.0f}x '
                   f'({points[0][0]:,} -> {points[-1][0]:,} tokens)')
    report.line('Does decode pay for the prompt?', [p[0] for p in points],
                {'p50 seconds': [p[1] for p in points]},
                xlabel='prompt tokens', ylabel='p50 seconds',
                note='Growing with length = decode re-prefills. Flat = blocks moved.')


## 10. See it live in Grafana

Every number above is one scrape. These links are the same metrics over
time, which is what distinguishes a steady state from a spike you happened
to catch. Requires the tunnel (`make ssh`) and `make observability`.


In [ ]:
report.heading('10. Grafana')
report.links([
    ('vLLM — KV, running/waiting, TTFT, prefix cache', grafana_dashboard('vllm')),
    ('Gateway — admit, shed, place, latency by stage', grafana_dashboard('gateway')),
    ('Router — pick, sticky, KV hop',                  grafana_dashboard('router')),
    ('KEDA — scaler value vs replicas',                grafana_dashboard('keda')),
    ('HAMi — device memory vs the slice cap',          grafana_dashboard('hami')),
    ('explore: KV usage per pod',
     grafana_explore('vllm:kv_cache_usage_perc')),
    ('explore: queue depth per pod',
     grafana_explore('vllm:num_requests_waiting')),
    ('explore: gateway shed by reason',
     grafana_explore('sum by (reason) (orch_shed_total)')),
], note='Grafana password is printed by make observability.')


## Save the report


In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
path = report.save(f'sanity-{stamp}')
fails = report.failures()
print('wrote', path)
print()
if fails:
    print('FAILURES — fix these before running load:')
    for f in fails: print('  -', f)
else:
    print('All checks passed. Next:  make load')
